# Table 1 notebook

This notebook is used to create the Table 1, which lists all genera of accepted 
names of the checklist, and the count of number of species.

1. Read file

In [1]:
import pandas as pd
import os

df_species = pd.read_csv('../dados_raw/checklist-species-29Abr2026.csv')

# Filter for taxonRank not genus
df_species.shape

(4440, 19)

2. Select accepted taxa

In [2]:
df_species_accepted = df_species[df_species['taxonomicStatus'].str.lower() == 'accepted'].reset_index(drop=True)
df_species_accepted.shape

(2012, 19)

3. Create a list of genus, and count the number of species for each genus

In [3]:
# from df_species_accepted create genus table with hierarchical counts in names
df = df_species_accepted.copy()
df[['class','order','family']] = df[['class','order','family']].fillna('Unknown')

# count scientificName per genus (within its class/order/family)
genus_counts = (
    df.groupby(['class','order','family','genus'], dropna=False)['scientificName']
      .nunique()
      .reset_index(name='count_species')
)

# counts for annotations
orders_per_class = df.groupby('class', dropna=False)['order'].nunique().reset_index(name='n_orders')
families_per_order = df.groupby(['class','order'], dropna=False)['family'].nunique().reset_index(name='n_families')
genera_per_family = df.groupby(['class','order','family'], dropna=False)['genus'].nunique().reset_index(name='n_genera')

# merge annotation counts into genus table
tbl = genus_counts.merge(orders_per_class, on='class', how='left')
tbl = tbl.merge(families_per_order, on=['class','order'], how='left')
tbl = tbl.merge(genera_per_family, on=['class','order','family'], how='left')

# format names with counts in parentheses
tbl['class']  = tbl['class'].astype(str)  + ' (' + tbl['n_orders'].astype(int).astype(str) + ')'
tbl['order']  = tbl['order'].astype(str)  + ' (' + tbl['n_families'].astype(int).astype(str) + ')'
tbl['family'] = tbl['family'].astype(str) + ' (' + tbl['n_genera'].astype(int).astype(str) + ')'

# final table
df_genus_table = tbl[['class','order','family','genus','count_species']].sort_values(['class','order','family','genus']).reset_index(drop=True)

df_genus_table

,class,order,family,genus,count_species
0,Agaricomycetes (3),Agaricales (1),Hygrophoraceae (1),Lichenomphalia,2
1,Agaricomycetes (3),Atheliales (1),Atheliaceae (1),Athelia,1
2,Agaricomycetes (3),Corticiales (1),Corticiaceae (1),Marchandiomyces,1
3,Arthoniomycetes (1),Arthoniales (7),Arthoniaceae (8),Arthonia,31
4,Arthoniomycetes (1),Arthoniales (7),Arthoniaceae (8),Arthothelium,1
...,...,...,...,...,...
485,Sordariomycetes (5),Xylariales (1),Leptosilliaceae (1),Leptosillia,1
486,Tremellomycetes (2),Filobasidiales (1),Filobasidiaceae (2),Heterocephalacria,1
487,Tremellomycetes (2),Filobasidiales (1),Filobasidiaceae (2),Zyzygomyces,1
488,Tremellomycetes (2),Tremellales (2),Bulleraceae (1),Biatoropsis,1


In [ ]:
Remove repeated values in the hierarchical columns so each category appears only once (blanked on subsequent rows):

```markdown
Use the cleaned display table for reporting:

```python
display_df = df_genus_table.copy()
display_df['class']  = display_df['class'].mask(display_df['class'].duplicated())
display_df['order']  = display_df['order'].mask(display_df.duplicated(subset=['class','order']))
display_df['family'] = display_df['family'].mask(display_df.duplicated(subset=['class','order','family']))
display_df
```
```

In [4]:
# Save the genus table as CSV in ../dados_process (filename: table2.csv):

df_genus_table.to_csv('../dados_process/table2.csv', index=False)